# Run + environment plot

Loads `{runname}_frames.csv` (produced by `fits_reprocess.py`) and the temperature/humidity time series for the same window, builds the conference figure (3 stacked panels), and runs the rolling-range event detection.

All distances are in **pixels** (no arcsec conversion here; the pixel scale is camera-dependent and lives in `pixel_scales.csv`).

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import ndimage as ndi

PROJECT_ROOT = Path.cwd()
TEMP_DIR = PROJECT_ROOT / 'temperature'
assert TEMP_DIR.is_dir(), f'temperature/ not found at {TEMP_DIR} -- run notebook from repo root'
sys.path.insert(0, str(TEMP_DIR))
import temp_functions as tf
TEMP_DAILY = TEMP_DIR / 'daily'  # parent temperature/ folder has a NaT-corrupt file; daily/ is clean

## Configuration

In [ ]:
RUN_PATH = Path('E:/Reverse Telescope Test Data/20260415_data/zoeystatic')

# Single-dot filter. Tested: main_camera runs satisfy n_peaks==1 cleanly. Dalsa Genie
# runs systematically project to multi-peak Y profiles (likely an optical artifact, not
# real two-dot data), so set this False when working with Genie runs to keep the
# centroid from the dominant-peak Gaussian fit.
FILTER_MULTI_PEAK = True

EVENT_WINDOW = '1h'   # rolling window length for event detection

# Thresholds picked to flag roughly the upper decile of 1h rolling-range windows on
# stable lab data (April zoeystatic). Re-tune per run if needed -- check the printed
# percentiles after running. Position thresholds in pixels; temp in deg C; humidity %RH.
EVENT_THRESHOLDS = {
    'position_px':  15.0,
    'temperature_C': 0.4,
    'humidity_pct':  3.5,
}
EVENT_SENSORS = {
    'temperature': 'SHT_Temperature_C',     # which sensor drives temp events
    'humidity':    'SHT_Relative_Humidity', # which drives humidity events
}
ENV_RESAMPLE = '30s'  # resample temp/humidity for plotting (raw is 1Hz)

## Functions

In [ ]:
def load_frames(run_path, filter_multi_peak=True):
    """Load per-frame CSV produced by fits_reprocess.py. Filters to fit_ok frames,
    optionally to single-dot frames, and subtracts first-frame position so X/Y
    start at zero. Raises if the filter leaves no rows."""
    runname = run_path.name
    csv_path = run_path / f'{runname}_frames.csv'
    df = pd.read_csv(csv_path, parse_dates=['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    n_total = len(df)
    keep = df['fit_ok'].copy()
    if filter_multi_peak:
        keep &= (df['n_peaks_x'] == 1) & (df['n_peaks_y'] == 1)
    n_multi = int(((df['n_peaks_x'] > 1) | (df['n_peaks_y'] > 1)).sum())
    n_failed = int((~df['fit_ok']).sum())
    df = df[keep].copy()
    if df.empty:
        raise ValueError(
            f'No frames survive filter for {runname}. '
            f'fit_ok={n_total - n_failed:,}/{n_total:,}, multi-peak frames={n_multi:,}. '
            f'Try setting FILTER_MULTI_PEAK = False if this is a Genie run.'
        )
    df['mu_x_rel'] = df['mu_x'] - df['mu_x'].iloc[0]
    df['mu_y_rel'] = df['mu_y'] - df['mu_y'].iloc[0]
    df = df.set_index('timestamp')
    kept_msg = f'  loaded {len(df):,} of {n_total:,} frames'
    if filter_multi_peak:
        kept_msg += f' (filtered out {n_multi:,} multi-peak, {n_failed:,} failed fits)'
    else:
        kept_msg += f' (multi-peak filter OFF; {n_failed:,} failed fits dropped)'
    print(kept_msg)
    return df

def load_environment(start, end, resample=ENV_RESAMPLE):
    return tf.builder(start, end, source_dir=str(TEMP_DAILY), resample_freq=resample)

In [ ]:
def detect_events(series, window='1h', range_threshold=1.0):
    """Return list of (start, end) intervals where the rolling-window range
    (max - min) exceeds range_threshold. Contiguous trigger runs are collapsed
    into single events."""
    s = series.dropna().sort_index()
    if s.empty:
        return []
    win = s.rolling(window)
    rolling_range = win.max() - win.min()
    above = (rolling_range > range_threshold).fillna(False).to_numpy()
    labels, n = ndi.label(above)
    out = []
    for k in range(1, n + 1):
        idx = np.where(labels == k)[0]
        out.append((s.index[idx[0]], s.index[idx[-1]]))
    return out

def intervals_overlap(a_start, a_end, b_start, b_end):
    return a_start <= b_end and b_start <= a_end

def count_overlaps(events_a, events_b):
    """Count how many a-events overlap any b-event (binary per a)."""
    n = 0
    for a in events_a:
        if any(intervals_overlap(a[0], a[1], b[0], b[1]) for b in events_b):
            n += 1
    return n

def event_duration_h(events):
    return sum((e[1] - e[0]).total_seconds() for e in events) / 3600.0

In [ ]:
def plot_run_with_environment(frames, env, events=None, title=None, out_path=None):
    """Three stacked panels with shared time x-axis:
      1. X/Y centroid drift (pixels)
      2. Temperature (all sensors)
      3. Humidity (all sensors)
    Optional `events` dict highlights event windows per panel."""
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True,
                              gridspec_kw={'hspace': 0.08})
    ax_pos, ax_t, ax_h = axes

    ax_pos.plot(frames.index, frames['mu_x_rel'], color='C0', label='X', lw=1.0)
    ax_pos.plot(frames.index, frames['mu_y_rel'], color='C2', label='Y', lw=1.0)
    ax_pos.axhline(0, color='0.5', lw=0.5)
    ax_pos.set_ylabel('Centroid drift (pixels)')
    ax_pos.legend(loc='upper right')

    for col, color in [('SHT_Temperature_C','C3'),('MCP_Temperature_C','C1'),('HDC_Temperature_C','C4')]:
        if col in env.columns and env[col].notna().any():
            ax_t.plot(env.index, env[col], color=color, label=col.split('_')[0], lw=1.0)
    ax_t.set_ylabel(r'Temperature ($^\circ$C)')
    ax_t.legend(loc='upper right')

    for col, color in [('SHT_Relative_Humidity','C3'),('HDC_Relative_Humidity','C4')]:
        if col in env.columns and env[col].notna().any():
            ax_h.plot(env.index, env[col], color=color, label=col.split('_')[0], lw=1.0)
    ax_h.set_ylabel('Relative humidity (%)')
    ax_h.legend(loc='upper right')

    if events is not None:
        _overlay_events(ax_pos, events.get('position', []), color='C0', alpha=0.15)
        _overlay_events(ax_t,   events.get('temperature', []), color='C3', alpha=0.15)
        _overlay_events(ax_h,   events.get('humidity', []), color='C4', alpha=0.15)
        _mark_joint(axes, events)

    for ax in axes:
        ax.xaxis.set_major_locator(mdates.DayLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
        ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))
        ax.grid(which='major', linestyle='-', alpha=0.5)
        ax.grid(which='minor', linestyle='--', alpha=0.25)
    for label in ax_h.get_xticklabels():
        label.set_rotation(45)
        label.set_horizontalalignment('right')
    ax_h.set_xlabel('Time')
    if title:
        fig.suptitle(title, y=0.995)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        print(f'  saved figure to {out_path}')
    return fig, axes

def _overlay_events(ax, events, color, alpha):
    for start, end in events:
        ax.axvspan(start, end, color=color, alpha=alpha, linewidth=0)

def _mark_joint(axes, events):
    pos_evts = events.get('position', [])
    env_evts = events.get('temperature', []) + events.get('humidity', [])
    for ps, pe in pos_evts:
        for es, ee in env_evts:
            if intervals_overlap(ps, pe, es, ee):
                mid = ps + (pe - ps) / 2
                for ax in axes:
                    ax.axvline(mid, color='k', linestyle=':', alpha=0.5, lw=0.8)
                break

In [ ]:
def summarize_events(events_dict, run_duration_h, runname='run', window=EVENT_WINDOW):
    series_names = list(events_dict.keys())
    print(f'\nRun: {runname}  (duration {run_duration_h:.1f} h, event window {window})')
    print('Event counts:')
    for name in series_names:
        evts = events_dict[name]
        dur = event_duration_h(evts)
        pct = 100 * dur / run_duration_h if run_duration_h else 0
        print(f'  {name:<12} {len(evts):>4}  ({dur:5.1f} h, {pct:4.1f}% of run)')
    print('\nCo-occurrence (row events overlapping any col event, count):')
    col_hdr = ' ' * 14 + ' '.join(f'{n:>10}' for n in series_names)
    print(col_hdr)
    for r in series_names:
        cells = ' '.join(f'{count_overlaps(events_dict[r], events_dict[c]):>10}' for c in series_names)
        print(f'  {r:<12} {cells}')

    pos = events_dict.get('X_drift', []) + events_dict.get('Y_drift', [])
    env = events_dict.get('temperature', []) + events_dict.get('humidity', [])
    pos_only = sum(1 for p in pos if not any(intervals_overlap(p[0], p[1], e[0], e[1]) for e in env))
    env_only = sum(1 for e in env if not any(intervals_overlap(e[0], e[1], p[0], p[1]) for p in pos))
    joint   = sum(1 for p in pos if any(intervals_overlap(p[0], p[1], e[0], e[1]) for e in env))
    print(f'\nPosition events with NO env co-event:  {pos_only} of {len(pos)}')
    print(f'Env events with NO position co-event:  {env_only} of {len(env)}')
    print(f'Joint events (position AND env):       {joint}')

def save_events_csv(events_dict, run_path):
    runname = run_path.name
    rows = []
    for series, evts in events_dict.items():
        for start, end in evts:
            rows.append({'series': series, 'start': start, 'end': end,
                         'duration_min': (end - start).total_seconds() / 60.0})
    out = run_path / f'{runname}_events.csv'
    pd.DataFrame(rows).to_csv(out, index=False)
    print(f'  events saved to {out}')
    return out

## Run it

In [ ]:
frames = load_frames(RUN_PATH, filter_multi_peak=FILTER_MULTI_PEAK)
start, end = frames.index.min(), frames.index.max()
run_h = (end - start).total_seconds() / 3600.0
print(f'  run window: {start}  ->  {end}  ({run_h:.1f} h)')
env = load_environment(start, end)
print(f'  env rows: {len(env):,}  cols: {list(env.columns)}')

In [ ]:
events = {
    'X_drift':     detect_events(frames['mu_x_rel'], EVENT_WINDOW, EVENT_THRESHOLDS['position_px']),
    'Y_drift':     detect_events(frames['mu_y_rel'], EVENT_WINDOW, EVENT_THRESHOLDS['position_px']),
    'temperature': detect_events(env[EVENT_SENSORS['temperature']], EVENT_WINDOW, EVENT_THRESHOLDS['temperature_C']),
    'humidity':    detect_events(env[EVENT_SENSORS['humidity']], EVENT_WINDOW, EVENT_THRESHOLDS['humidity_pct']),
}
summarize_events(events, run_h, runname=RUN_PATH.name)

In [ ]:
events_for_plot = {
    'position':    events['X_drift'] + events['Y_drift'],
    'temperature': events['temperature'],
    'humidity':    events['humidity'],
}
out_png = RUN_PATH / f'{RUN_PATH.name}_environment.png'
fig, axes = plot_run_with_environment(
    frames, env, events=events_for_plot,
    title=f'{RUN_PATH.name}  ({start:%Y-%m-%d %H:%M} -> {end:%Y-%m-%d %H:%M})',
    out_path=out_png,
)
save_events_csv(events, RUN_PATH)
plt.show()

## Cross-run summary (concatenate all `_events.csv` files)

In [ ]:
def summarize_all_runs(root='E:/Reverse Telescope Test Data'):
    root = Path(root)
    rows = []
    for events_csv in sorted(root.glob('*_data/*/*_events.csv')):
        df = pd.read_csv(events_csv, parse_dates=['start', 'end'])
        df['runname'] = events_csv.stem.replace('_events', '')
        rows.append(df)
    if not rows:
        print('No _events.csv files yet.'); return None
    all_events = pd.concat(rows, ignore_index=True)
    counts = all_events.groupby(['runname', 'series']).size().unstack(fill_value=0)
    print(counts)
    return all_events

# summarize_all_runs()